# Oxford 전체 이미지·품종·마스크 연결 확인
metadata.csv의 한 행은 사진 한 장입니다. image_path로 사진, mask_path로 해당 사진의 영역 정답을 엽니다.
원본 픽셀과 채널은 보존했습니다. 모델 입력에서는 RGB 변환 및 모델에 맞는 전처리를 적용하세요.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Rectangle
from IPython.display import display

DATA_DIR = Path.cwd()
if not (DATA_DIR / "metadata.csv").exists() and (DATA_DIR / "oxford_dataset" / "metadata.csv").exists():
    DATA_DIR = DATA_DIR / "oxford_dataset"
# 필요할 때만 실제 생성 폴더 경로로 바꾸세요.
# DATA_DIR = Path(r"C:/Users/jihun/Downloads/Oxford/oxford_dataset")
if not (DATA_DIR / "metadata.csv").exists():
    raise FileNotFoundError("DATA_DIR을 metadata.csv가 있는 oxford_dataset 폴더로 지정하세요.")
metadata = pd.read_csv(DATA_DIR / "metadata.csv")
classes = pd.read_csv(DATA_DIR / "classes.csv")
head_boxes = pd.read_csv(DATA_DIR / "head_boxes.csv")
print("전체 이미지:", len(metadata))
display(pd.crosstab(metadata["class_name"], metadata["split"]))
display(classes.head())

## 1. 사진 한 장과 라벨 확인

In [ ]:
row = metadata.iloc[0]
print("파일:", row["file_name"])
print("품종:", row["class_name"], "학습 라벨:", row["label"])
print("동물:", row["species_name"], "분할:", row["split"])
with Image.open(DATA_DIR / row["image_path"]) as source:
    image = source.convert("RGB")
display(image)

## 2. 같은 사진의 영역 정답 확인
학습용 masks PNG: 0=배경, 1=동물, 2=미분류/불확실 영역.
원본 trimap의 1=동물, 2=배경, 3=미분류를 변환했습니다.
일반 사진 뷰어에서는 값이 0~2라 거의 검게 보입니다. 아래처럼 색을 지정해서 확인하세요.
마스크 크기를 변경할 때는 최근접 보간을 사용하세요.

In [ ]:
with Image.open(DATA_DIR / row["mask_path"]) as source:
    mask = np.asarray(source).copy()
colors = ListedColormap(["black", "limegreen", "orange"])
norm = BoundaryNorm([-0.5, 0.5, 1.5, 2.5], colors.N)
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image); axes[0].set_title("Image")
axes[1].imshow(mask, cmap=colors, norm=norm)
axes[1].set_title("0 background / 1 pet / 2 unknown")
axes[2].imshow(image)
axes[2].imshow(np.ma.masked_where(mask == 0, mask), cmap=colors, norm=norm, alpha=0.45)
axes[2].set_title("Overlay")
for ax in axes: ax.axis("off")
plt.show()
print("마스크 값:", np.unique(mask))

## 3. 머리 박스가 있는 이미지 확인
박스는 머리 영역입니다. 동물 전신 박스가 아닙니다. 박스가 없는 사진도 분류와 세그멘테이션에 사용할 수 있습니다.

In [ ]:
if head_boxes.empty:
    print("제공된 XML 머리 박스가 없습니다.")
else:
    box = head_boxes.iloc[0]
    with Image.open(DATA_DIR / box["image_path"]) as source:
        box_image = source.convert("RGB")
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(box_image)
    for _, b in head_boxes.loc[head_boxes["image_id"] == box["image_id"]].iterrows():
        ax.add_patch(Rectangle((b["xmin"], b["ymin"]), b["xmax"]-b["xmin"], b["ymax"]-b["ymin"],
                               fill=False, edgecolor="red", linewidth=2))
    ax.set_title("Head bounding box")
    ax.axis("off")
    plt.show()

## 4. 학습·검증·평가 자료 불러오기
공식 test는 그대로 유지합니다. 공식 trainval 안에서만 품종별 80:20으로 train/val을 나눴습니다.

In [ ]:
train = pd.read_csv(DATA_DIR / "train.csv")
val = pd.read_csv(DATA_DIR / "val.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
assert set(train.image_id).isdisjoint(val.image_id)
assert set(train.image_id).isdisjoint(test.image_id)
assert set(val.image_id).isdisjoint(test.image_id)
assert len(train) + len(val) + len(test) == len(metadata)
print("Train:", len(train), "Validation:", len(val), "Test:", len(test))
print("image_id 기준 분할 간 중복 없음")
display(train[["image_path", "label", "class_name", "species_label", "mask_path"]].head())

## 학습할 때의 기준
- 37개 품종 분류: label (0~36).
- 고양이/개 분류: species_label (0=cat, 1=dog).
- class_id는 원본 1~37, breed_id_within_species는 종별 번호이므로 서로 구분하세요.
- 세그멘테이션: mask_path. 원본 trimap_path도 별도로 보존합니다.
- 미분류 영역(2)을 배경으로 간주하지 마세요. 3클래스로 학습하거나, 동물/배경 이진 학습에서 제외하세요.
- masks 이미지에 /255 정규화를 적용하지 마세요. 정수 라벨입니다.
- 증강은 학습 자료에만 적용합니다. 분할 증강에서는 이미지/마스크에 같은 공간 변환을 적용하세요.
- 입력 폴더는 images/train/00_Abyssinian 같은 구조입니다. 숫자 접두사는 폴더 정렬 순서와 label을 맞추기 위한 것입니다.
- 학습은 train, 설정 선택은 val, 최종 평가는 test를 사용하세요.
